# 03 — Two-Stage Default · REG · xgb

Stage 2 회귀 (y>0 only conditional, `E[Y|Y>0,x]`) → die-level reg_pred csv → combine 단계에서 clf prob과 곱.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/reg/xgb/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` (strategy_common §1)
- **TARGET_TRANSFORM**: `'none'` 고정 (strategy_common §24 — log1p_check 검증)
- **HPO**: N_TRIALS=1, **anchor enqueue + Narrow ±30%**
  - 1차 anchor는 log1p ON 컨텍스트지만 시작점으로 활용 — Optuna가 OFF 환경에 맞게 재탐색
- **Sampler/Pruner/Timeout**: §4·§25


## 1. 환경 설정 + import

In [1]:
import os, sys

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'  # ★ Colab 사용 시 신규 modeling.zip ID 입력
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip (RESUME용)
RESUME                 = True   # True: 기존 db에 이어서 / False: 처음부터 (db 있으면 에러)

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    # ── RESUME: 기존 4_output 복원 (Colab) ──
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, models
from meta_features import add_meta_features   # 2_preprocessing/meta_features.py — 2026-05-09 결정 반영

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
optuna v4.7.0


## 2. 실험 설정

In [2]:
# ── 모델 ──
REG_MODEL_NAME = 'xgb'
assert REG_MODEL_NAME in models.AVAILABLE_MODELS

# ── 실험 식별 ──
EXP_ID   = f'ts-reg-{REG_MODEL_NAME}-002'
EXP_MEMO = f'Two-Stage default · REG · {REG_MODEL_NAME} · y>0 conditional · transform=none'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS         = 3000
N_FOLDS          = 5
N_STARTUP_TRIALS = 50
N_JOBS           = -1   # 가용 코어 전부
TIMEOUT_SEC      = 20 * 60 * 60  # ★ §25

# ── Two-Stage Stage 2 정책 ──
TARGET_TRANSFORM = 'none'   # ★ strategy_common §24 (log1p_check 검증: none=log1p 동등)
Y_POSITIVE_ONLY  = True
CLIP_Y_EXTREME   = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'reg', REG_MODEL_NAME, EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 트리 PP_FIXED ──
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ── 1차 best anchor (strategy.md §4.3, log1p ON 컨텍스트) — log1p OFF에서 재탐색 시작점 ──
REG_ANCHOR = {'n_estimators': 1423, 'learning_rate': 0.0363, 'max_depth': 10, 'min_child_weight': 0.621, 'subsample': 0.728, 'colsample_bytree': 0.618, 'reg_alpha': 0.0168, 'reg_lambda': 3.89e-06, 'gamma': 3.84e-06}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'REG_MODEL_NAME: {REG_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | Y_POSITIVE_ONLY={Y_POSITIVE_ONLY}')
print(f'OUT_DIR={OUT_DIR}')


EXP: ts-reg-xgb-002 | USER: jh
REG_MODEL_NAME: xgb
N_TRIALS=1 | N_FOLDS=5 | N_JOBS=7 | TIMEOUT_SEC=None
TARGET_TRANSFORM=none | Y_POSITIVE_ONLY=True
OUT_DIR=C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\reg\xgb


## 3. 데이터 로드 + Y clip + 전처리 (target_transform=none)

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# target transform: 'none' 고정 (strategy_common §24)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24 — 트리 target_transform=none 통일)')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# ── 메타피처 추가 (2026-05-09 결정: 트리=position raw + die_xy continuous) ──
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

yt = ys_input['train'][TARGET_COL]
print(f'\nUnit y>0 비율 (train): {(yt > 0).mean():.4f} ({(yt > 0).sum():,} unit)')
print(f'E[Y | Y>0]            = {yt[yt > 0].mean():.6f}')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] none (strategy_common §24 — 트리 target_transform=none 통일)
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031


[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개


...


  [요약] 343,494 → 공간(161,870) → lot(100,428) → 전체(81,196) → 잔여(0)


[고상관 제거] threshold=0.96, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.96
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 633)

클리닝 완료: 1031 → 564 features (467개 제거)
  + indicator 컬럼: 9개 → 총 573개
  train: (104748, 633)
  val:   (34908, 633)
  test:  (34916, 633)
이상치 처리 파이프라인 시작 (method=winsorize)


[이상치 탐지] IQR × 1.5
  이상치 > 5%: 112개
  이상치 > 10%: 64개


[Winsorization] lower=0%, upper=99%
  적용 feature: 573개

이상치 처리 완료 (method=winsorize)
  train: (104748, 633)


[add_meta_features] position_mode='raw', use_die_xy=True, use_loc_x_ohe=False → position=['position'], die_xy=['die_x', 'die_y'] (feat_cols: 576)

[전처리 완료] feat_cols: 576

Unit y>0 비율 (train): 0.2920 (7,646 unit)
E[Y | Y>0]            = 0.008496


## 4. Optuna HPO (REG, y>0 die만 학습)

- `run_hpo(y_positive_only=True, target_transform_fn=None, target_inverse_fn=None)`
- **Sampler/Pruner/Timeout**: §4·§25
- **anchor enqueue**: 1차 best HP (log1p ON 컨텍스트) — 시작점, Optuna가 재탐색
- 손실함수는 anchor에서 빼고 Optuna가 재선택 (log1p OFF 환경)

In [4]:
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'reg_model_name':   REG_MODEL_NAME,
    'target_transform': TARGET_TRANSFORM,
    'y_positive_only':  Y_POSITIVE_ONLY,
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'n_jobs':           N_JOBS,
    'timeout_sec':      TIMEOUT_SEC,
    'seed':             SEED,
    'anchor':           REG_ANCHOR,
    'sampler':          'TPE seed=None multivariate group',
    'pruner':           f'MedianPruner n_warmup=10',
}

sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
pruner  = MedianPruner(n_warmup_steps=10)

res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    user_attrs=study_meta,
    sampler=sampler,
    pruner=pruner,
    enqueue_trials=[REG_ANCHOR],   # ★ anchor 첫 trial 강제 (§5)
    timeout=TIMEOUT_SEC,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']

print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'best_params = {best_params}')


[enqueue] 1 anchor trial(s) 강제


  0%|          | 0/1 [00:00<?, ?it/s]


[HPO 완료] best OOF RMSE = 0.007567
best_params = {'n_estimators': 1423, 'learning_rate': 0.0363, 'max_depth': 10, 'min_child_weight': 0.621, 'subsample': 0.728, 'colsample_bytree': 0.618, 'reg_alpha': 0.0168, 'reg_lambda': 3.89e-06, 'gamma': 3.84e-06, 'objective': 'count:poisson'}


## 5. Best trial 재학습 (K-fold OOF) + die-level reg_pred 캐쳐

In [5]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

y_train_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_true   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_unit  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_train_true.index]
val_unit  = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
test_unit = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.values  - y_train_true.values) ** 2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.values  - y_val_true.values)   ** 2)))
test_rmse = float(np.sqrt(np.mean((test_unit.values - y_test_true.values)  ** 2)))

print(f'\n[Refit 완료] (reg 단독, y>0 conditional, transform=none)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')


[refit fold 1/5] tr_units=20949, vl_units=5238


[refit fold 2/5] tr_units=20949, vl_units=5238


[refit fold 3/5] tr_units=20950, vl_units=5237


[refit fold 4/5] tr_units=20950, vl_units=5237


[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료] (reg 단독, y>0 conditional, transform=none)
  OOF  unit RMSE = 0.007567
  val  unit RMSE = 0.007686
  test unit RMSE = 0.009816


## 6. 산출물 저장

In [6]:
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=None,   # ★ combine 단계에서 후처리 (여기선 단순 mean)
    study_meta=study_meta,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'reg_{REG_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass


[save_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\reg\xgb 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=mean)
  best_params.json                      10.0 KB
  fold_models.pkl                   60,396.3 KB
  oof_die.csv                        5,518.9 KB
  oof_unit.csv                         954.1 KB
  optuna_jh_ts-reg-xgb-002.db          112.0 KB
  test_die.csv                       1,839.6 KB
  test_unit.csv                        318.0 KB
  val_die.csv                        1,839.8 KB
  val_unit.csv                         318.0 KB
